In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

ruta_brutos = '/content/drive/MyDrive/IDS2018_brutos'
ruta_salida = '/content/drive/MyDrive/IDS2018_procesados'
os.makedirs(ruta_salida, exist_ok=True)

print("Configuracion lista.")
print(f"Origen : {ruta_brutos}")
print(f"Destino: {ruta_salida}")

Mounted at /content/drive
Configuracion lista.
Origen : /content/drive/MyDrive/IDS2018_brutos
Destino: /content/drive/MyDrive/IDS2018_procesados


In [ ]:
archivos = {
    "archivo1_bruto.csv": "Fuerza bruta FTP",
    "archivo2_bruto.csv": "DoS GoldenEye + Slowloris",
    "archivo3_bruto.csv": "DDoS HOIC + LOIC-UDP",
    "archivo4_bruto.csv": "Web attacks XSS + SQL",
    "archivo5_bruto.csv": "Botnet ARES",
}

print("="*60)
print("CARGANDO Y UNIENDO 5 ARCHIVOS")
print("="*60)

dfs = []
for archivo, categoria in archivos.items():
    ruta = f"{ruta_brutos}/{archivo}"
    df_temp = pd.read_csv(ruta, low_memory=False)
    print(f"  {archivo} ({categoria}): {df_temp.shape}")
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
print(f"\nDataset unificado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

# Distribución inicial de clases
print("\nDistribucion inicial de clases:")
print(df['Label'].value_counts().to_string())

CARGANDO Y UNIENDO 5 ARCHIVOS
  archivo1_bruto.csv (Fuerza bruta FTP): (1048575, 80)
  archivo2_bruto.csv (DoS GoldenEye + Slowloris): (1048575, 80)
  archivo3_bruto.csv (DDoS HOIC + LOIC-UDP): (1048575, 80)
  archivo4_bruto.csv (Web attacks XSS + SQL): (1048575, 80)
  archivo5_bruto.csv (Botnet ARES): (1048575, 80)

Dataset unificado: 5,242,875 filas x 80 columnas

Distribucion inicial de clases:
Label
Benign                   3835133
DDOS attack-HOIC          686012
Bot                       286191
FTP-BruteForce            193360
SSH-Bruteforce            187589
DoS attacks-GoldenEye      41508
DoS attacks-Slowloris      10990
DDOS attack-LOIC-UDP        1730
Brute Force -Web             249
Brute Force -XSS              79
SQL Injection                 34


In [ ]:
# ── CELDA 3: Limpieza de datos ───────────────────────────────
# Respaldo: Park et al. (2025), Aljuaid & Alshamrani (2024),
#           Almuhanna & Dardouri (2025)
print("="*60)
print("LIMPIEZA DE DATOS")
print("="*60)

filas_inicial = len(df)

# Reemplazar infinitos por NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f"Infinitos reemplazados por NaN")

# Eliminar filas con valores nulos
nulos = df.isna().sum().sum()
df.dropna(inplace=True)
print(f"Filas eliminadas por nulos: {filas_inicial - len(df):,} ({nulos:,} valores nulos)")

# Eliminar duplicados
filas_antes_dup = len(df)
df.drop_duplicates(inplace=True)
print(f"Filas duplicadas eliminadas: {filas_antes_dup - len(df):,}")

# Eliminar columnas con >30% de ceros
# Respaldo: Advancing Software Security in Cloud Platforms (2024)
threshold_ceros = 0.30
cols_ceros = [col for col in df.columns
              if col != 'Label' and
              (df[col] == 0).sum() / len(df) > threshold_ceros]
df.drop(columns=cols_ceros, inplace=True)
print(f"Columnas eliminadas (>30% ceros): {len(cols_ceros)}")

# Eliminar columnas con varianza cero
# Respaldo: Al-Ghuwairi et al. (2023)
numeric_cols = df.select_dtypes(include=[np.number]).columns
cols_var_cero = numeric_cols[df[numeric_cols].var() == 0].tolist()
df.drop(columns=cols_var_cero, inplace=True)
print(f"Columnas eliminadas (varianza cero): {len(cols_var_cero)}")

print(f"\nDimensiones tras limpieza: {df.shape[0]:,} filas x {df.shape[1]} columnas")

LIMPIEZA DE DATOS
Infinitos reemplazados por NaN
Filas eliminadas por nulos: 21,511 (43,022 valores nulos)
Filas duplicadas eliminadas: 254,323
Columnas eliminadas (>30% ceros): 56
Columnas eliminadas (varianza cero): 0

Dimensiones tras limpieza: 4,967,041 filas x 24 columnas


In [ ]:
# ── CELDA 4: Separar etiquetas ───────────────────────────────
# Respaldo: Park et al. (2025)
print("="*60)
print("SEPARACION DE ETIQUETAS")
print("="*60)

# Guardar etiquetas antes de normalizar
labels = df['Label'].copy()
df_features = df.drop(columns=['Label'])

print(f"Features: {df_features.shape[1]} columnas")
print(f"Etiquetas guardadas: {labels.value_counts().to_string()}")

SEPARACION DE ETIQUETAS
Features: 23 columnas
Etiquetas guardadas: Label
Benign                   3805770
DDOS attack-HOIC          668461
Bot                       282310
SSH-Bruteforce            117322
DoS attacks-GoldenEye      41455
FTP-BruteForce             39346
DoS attacks-Slowloris      10285
DDOS attack-LOIC-UDP        1730
Brute Force -Web             249
Brute Force -XSS              79
SQL Injection                 34


In [ ]:
# ── CELDA 5: Normalización Min-Max ───────────────────────────
# Respaldo: Park et al. (2025),
#           Advancing Software Security in Cloud Platforms (2024)
from sklearn.preprocessing import MinMaxScaler

print("="*60)
print("NORMALIZACION MIN-MAX [0, 1]")
print("="*60)

# Solo columnas numéricas
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
scaler = MinMaxScaler()
df_features[numeric_cols] = scaler.fit_transform(df_features[numeric_cols])

print(f"Normalizacion aplicada a {len(numeric_cols)} columnas numericas")
print(f"Rango verificado — min: {df_features[numeric_cols].min().min():.4f}")
print(f"Rango verificado — max: {df_features[numeric_cols].max().max():.4f}")

NORMALIZACION MIN-MAX [0, 1]
Normalizacion aplicada a 22 columnas numericas
Rango verificado — min: 0.0000
Rango verificado — max: 1.0000


In [ ]:
# ── CELDA 6: Selección de características — Pearson ──────────
# Respaldo: Park et al. (2025) — umbral correlacion > 0.90
print("="*60)
print("SELECCION DE CARACTERISTICAS — CORRELACION PEARSON")
print("="*60)

cols_antes = df_features.shape[1]

# Calcular matriz de correlación
corr_matrix = df_features[numeric_cols].corr().abs()

# Identificar pares con correlación > 0.90
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
cols_alta_corr = [col for col in upper_triangle.columns
                  if any(upper_triangle[col] > 0.90)]

df_features.drop(columns=cols_alta_corr, inplace=True)
cols_despues = df_features.shape[1]

print(f"Columnas antes : {cols_antes}")
print(f"Columnas eliminadas (correlacion > 0.90): {len(cols_alta_corr)}")
print(f"Columnas finales: {cols_despues}")

SELECCION DE CARACTERISTICAS — CORRELACION PEARSON
Columnas antes : 23
Columnas eliminadas (correlacion > 0.90): 9
Columnas finales: 14


In [ ]:
# ── CELDA 7: Separar Training set y Test set ─────────────────
# Respaldo: Park et al. (2025)
# Training: SOLO tráfico benigno (sin etiquetas)
# Test: tráfico mixto (benigno + ataques, con etiquetas)
print("="*60)
print("SEPARACION TRAINING SET Y TEST SET")
print("="*60)

# Reconstruir dataframe con features + labels
df_completo = df_features.copy()
df_completo['Label'] = labels.values

# Training set — solo benigno, sin columna Label
mask_benigno = df_completo['Label'] == 'Benign'
df_train = df_completo[mask_benigno].drop(columns=['Label'])

# Test set — todo el dataset, con etiquetas separadas
df_test = df_completo.drop(columns=['Label'])
df_test_labels = df_completo['Label']

print(f"Training set (solo benigno) : {df_train.shape[0]:,} filas x {df_train.shape[1]} columnas")
print(f"Test set (mixto)            : {df_test.shape[0]:,} filas x {df_test.shape[1]} columnas")
print(f"\nDistribucion test set:")
print(df_test_labels.value_counts().to_string())

SEPARACION TRAINING SET Y TEST SET
Training set (solo benigno) : 3,805,770 filas x 14 columnas
Test set (mixto)            : 4,967,041 filas x 14 columnas

Distribucion test set:
Label
Benign                   3805770
DDOS attack-HOIC          668461
Bot                       282310
SSH-Bruteforce            117322
DoS attacks-GoldenEye      41455
FTP-BruteForce             39346
DoS attacks-Slowloris      10285
DDOS attack-LOIC-UDP        1730
Brute Force -Web             249
Brute Force -XSS              79
SQL Injection                 34


In [ ]:
# ── CELDA 8: Guardar en Drive ────────────────────────────────
print("="*60)
print("GUARDANDO ARCHIVOS EN DRIVE")
print("="*60)

# Training set
ruta_train = f"{ruta_salida}/dataset_train.csv"
df_train.to_csv(ruta_train, index=False)
size_train = os.path.getsize(ruta_train) / 1024 / 1024
print(f"dataset_train.csv     : {df_train.shape[0]:,} filas — {size_train:.1f} MB")

# Test set features
ruta_test = f"{ruta_salida}/dataset_test.csv"
df_test.to_csv(ruta_test, index=False)
size_test = os.path.getsize(ruta_test) / 1024 / 1024
print(f"dataset_test.csv      : {df_test.shape[0]:,} filas — {size_test:.1f} MB")

# Test set labels
ruta_labels = f"{ruta_salida}/dataset_test_labels.csv"
df_test_labels.to_csv(ruta_labels, index=False)
print(f"dataset_test_labels.csv: {len(df_test_labels):,} etiquetas")

print("\n" + "="*60)
print("RESUMEN FINAL DEL PREPROCESAMIENTO")
print("="*60)
print(f"Features finales        : {df_train.shape[1]} columnas")
print(f"Training set (benigno)  : {df_train.shape[0]:,} filas")
print(f"Test set (mixto)        : {df_test.shape[0]:,} filas")
print(f"Archivos guardados en   : {ruta_salida}")
print("\n✅ Preprocesamiento completado.")
print("Siguiente paso: Cuaderno 03 — Isolation Forest")

GUARDANDO ARCHIVOS EN DRIVE
dataset_train.csv     : 3,805,770 filas — 902.4 MB
dataset_test.csv      : 4,967,041 filas — 1176.4 MB
dataset_test_labels.csv: 4,967,041 etiquetas

RESUMEN FINAL DEL PREPROCESAMIENTO
Features finales        : 14 columnas
Training set (benigno)  : 3,805,770 filas
Test set (mixto)        : 4,967,041 filas
Archivos guardados en   : /content/drive/MyDrive/IDS2018_procesados

✅ Preprocesamiento completado.
Siguiente paso: Cuaderno 03 — Isolation Forest
